In [2]:
!pip install -q transformers datasets evaluate accelerate
!pip install -q torch torchvision torchaudio

In [ ]:
from datasets import load_dataset
data = load_dataset('allenai/sciq')

In [34]:
data['train'][0]

{'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?',
 'distractor3': 'viruses',
 'distractor1': 'protozoa',
 'distractor2': 'gymnosperms',
 'correct_answer': 'mesophilic organisms',
 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.'}

In [35]:
data['train'].shape

(11679, 6)

# Preprocessing the dataset

In [36]:
import random
def process(example):
    question = example['question']
    context =  example['support']
    choices = [
        example['correct_answer'],
        example['distractor1'],
        example['distractor2'],
        example['distractor3']
    ]
    correct = example['correct_answer']
    random.shuffle(choices) 
    # shuffles the options to get a random answer other wise model will look at the statistical posi
    label = choices.index(correct)
    return {
        "question": example["question"],
        "support": example["support"],
        "choices": choices,
        "label": label
    }


In [37]:
dataset = data.map(process)

In [38]:
dataset['train'][0]

{'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?',
 'distractor3': 'viruses',
 'distractor1': 'protozoa',
 'distractor2': 'gymnosperms',
 'correct_answer': 'mesophilic organisms',
 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.',
 'choices': ['mesophilic organisms', 'protozoa', 'viruses', 'gymnosperms'],
 'label': 0}

# Tokenizing the dataset

In [39]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    # 'roberta-base',
    'microsoft/deberta-v3-base'
)

In [40]:
def preprocess_function(examples):

    first_sentences = []
    second_sentences = []

    for q, support, choices in zip(
        examples["question"],
        examples["support"],
        examples["choices"]
    ):

        first_sentences.extend(
            [q] * 4
        )

        second_sentences.extend(
            [
                support + " [SEP] " + choice
                for choice in choices
            ]
        )

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    result = {
        k: [
            v[i:i+4]
            for i in range(0, len(v), 4)
        ]
        for k, v in tokenized.items()
    }

    result["labels"] = examples["label"]

    return result

In [41]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [42]:
sample = tokenized_dataset["train"][0]

print(sample.keys())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


In [43]:
len(sample["input_ids"])

4

In [44]:
sample['labels']

0

# Loading the Model

In [45]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained(
    "microsoft/deberta-v3-base",
    dtype=torch.float32
)

model = model.cuda()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                

In [46]:
sample = tokenized_dataset["train"][0]


In [47]:
import torch

batch = {
    "input_ids": torch.tensor(
        [sample["input_ids"]]
    ),
    "attention_mask": torch.tensor(
        [sample["attention_mask"]]
    ),
}

In [48]:
batch

{'input_ids': tensor([[[458, 810, 265,  ...,   0,   0,   0],
          [458, 810, 265,  ...,   0,   0,   0],
          [458, 810, 265,  ...,   0,   0,   0],
          [458, 810, 265,  ...,   0,   0,   0]]]),
 'attention_mask': tensor([[[1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0]]])}

# Data batches and starting prep

In [49]:
from dataclasses import dataclass
from typing import Optional, Union
import torch

@dataclass
class dataCollator:

    tokenizer: object
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):

        labels = [feature.pop("labels") for feature in features]

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened_features = []

        for feature in features:
            for i in range(num_choices):
                flattened_features.append({
                    k: v[i]
                    for k, v in feature.items()
                })

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt"
        )

        batch = {
            k: v.view(batch_size, num_choices, -1)
            for k, v in batch.items()
        }

        batch["labels"] = torch.tensor(labels)

        return batch

In [50]:
data_collator = dataCollator(
    tokenizer=tokenizer
)

In [51]:
batch = data_collator(
    [tokenized_dataset["train"][0],
     tokenized_dataset["train"][1]]
)

for k,v in batch.items():
    print(k, v.shape)

input_ids torch.Size([2, 4, 256])
token_type_ids torch.Size([2, 4, 256])
attention_mask torch.Size([2, 4, 256])
labels torch.Size([2])


In [52]:
!pip install -q evaluate

In [53]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [64]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert-sciq-mcqa",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=4,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=50,
    fp16=False,
    bf16=False,
    load_best_model_at_end=True,

    report_to="none"
)

In [65]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [66]:
device = next(model.parameters()).device

sample = data_collator([
    tokenized_dataset["train"][0],
    tokenized_dataset["train"][1]
])

sample = {
    k: v.to(device)
    for k, v in sample.items()
}

outputs = model(**sample)

print(outputs.loss)
print(outputs.logits.shape)

tensor(1.4467, device='cuda:0', grad_fn=<NllLossBackward0>)
torch.Size([2, 4])


In [57]:
print(sample["labels"])
print(sample["labels"].dtype)

tensor([0, 2], device='cuda:0')
torch.int64


In [58]:
labels = [x["labels"] for x in tokenized_dataset["train"]]

print(min(labels))
print(max(labels))
print(set(labels))

0
3
{0, 1, 2, 3}


In [59]:
print(sample["input_ids"][0][0][:20])
print(sample["attention_mask"][0][0][:20])

tensor([  458,   810,   265, 17370,   269,  4099,   427,   267,  3822,   265,
         3300,   405,   283,  3188,   263, 11645,   302, 56790, 45530,   268],
       device='cuda:0')
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       device='cuda:0')


In [60]:
print(torch.isnan(sample["input_ids"]).any())
print(torch.isnan(sample["attention_mask"]).any())

tensor(False, device='cuda:0')
tensor(False, device='cuda:0')


In [61]:
device = next(model.parameters()).device

batch = data_collator([
    tokenized_dataset["train"][0],
    tokenized_dataset["train"][1]
])

labels = batch.pop("labels")

batch = {k:v.to(device) for k,v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print(outputs.logits)
print(torch.isnan(outputs.logits).any())

tensor([[ 0.0876,  0.0827,  0.0818,  0.0923],
        [-0.0163, -0.0231, -0.0238, -0.0212]], device='cuda:0')
tensor(False, device='cuda:0')


In [62]:
labels = sample["labels"]

outputs = model(
    input_ids=sample["input_ids"],
    attention_mask=sample["attention_mask"],
    labels=labels
)

print(outputs.loss)

tensor(1.3869, device='cuda:0', grad_fn=<NllLossBackward0>)


In [67]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.415897,0.618186,0.902000
2,0.606920,0.655753,0.907000
3,0.233301,0.810960,0.903000
4,0.313513,0.807412,0.906000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=5840, training_loss=0.43117299177875257, metrics={'train_runtime': 6771.1822, 'train_samples_per_second': 6.899, 'train_steps_per_second': 0.862, 'total_flos': 2.458321227111629e+16, 'train_loss': 0.43117299177875257, 'epoch': 4.0})

In [68]:
trainer.save_model("./deberta_sciq_model")
tokenizer.save_pretrained("./deberta_sciq_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./deberta_sciq_model/tokenizer_config.json',
 './deberta_sciq_model/tokenizer.json')